<a href="https://colab.research.google.com/github/SehrishbAsghar/FlyRank_ML_Internship_Sehrish/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

raw = con.sql(f"""
    SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
""").df()

content = con.sql(f"""
    SELECT content_hash_id, content_type
    FROM read_parquet('{rel}/dim_content.parquet')
""").df()

df = raw.merge(content, on="content_hash_id", how="left")
df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"].replace(0, np.nan)
df["ctr"] = df["ctr"].fillna(0)

def position_tier(pos):
    if pd.isna(pos): return "unknown"
    elif pos <= 3: return "top_3"
    elif pos <= 10: return "page_1"
    elif pos <= 20: return "striking"
    elif pos <= 50: return "page_3_5"
    else: return "deep"

df["position_tier"] = df["gsc_avg_position"].apply(position_tier)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [2]:
signal1 = df.groupby("position_tier")["ctr"].agg(["mean", "count"]).round(4)
print(signal1)


                 mean    count
position_tier                 
deep           0.0005   276863
page_1         0.0035  1456122
page_3_5       0.0016   631491
striking       0.0028   519223
top_3          0.0048   727362


In [3]:
df["volume_bucket"] = pd.qcut(df["gsc_impressions"], q=4, labels=["low", "mid_low", "mid_high", "high"], duplicates="drop")
signal2 = df.groupby("volume_bucket", observed=True)["ctr"].agg(["mean", "count"]).round(4)
print(signal2)


                 mean   count
volume_bucket                
low            0.0041  973112
mid_low        0.0024  868861
mid_high       0.0027  873000
high           0.0031  896088


# **Signal checks**

Signal 1: CTR vs. position_tier(behind the CTR-fix flag logic)

Verdict: CONFIRMED.
CTR declines monotonically and cleanly as position worsens, across large sample sizes. This is a trustworthy signal to base a rule on.

Signal 2: CTR vs. volume
(behind the quick-win logic)

Verdict: MIXED. No monotonic relationship between raw impression volume and
CTR. No monotonic relationship between raw impression volume and
CTR

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The rule in plain words:

Flag a page as an opportunity if its CTR is meaningfully below what's expected for its position tier, AND it has enough impression volume that fixing it would actually matter. Position tier drives whether a page should be flagged (Signal 1, confirmed); impression volume scales how much it matters if flagged, since volume itself doesn't predict CTR
level (Signal 2, mixed); it just tells us the size of the prize if we fix it.

Concretely: compute each page's CTR gap versus its position_tier's average CTR. If the gap is negative (below-tier-average) and impressions are above the median,
flag it as an opportunity, with a score = (tier_avg_ctr - page_ctr) x
log(impressions)

So a bigger gap on a higher-volume page scores higher than
the same gap on a low-volume page.

Reason codes the rule can output:

- `LOW_CTR_HIGH_VOLUME` CTR below tier average, high impressions: the core flagged opportunity, biggest fix-it-first candidate.
- `LOW_CTR_LOW_VOLUME` CTR below tier average, but low impressions: real gap, but low absolute upside; deprioritized, not ignored.
- `AT_OR_ABOVE_EXPECTED`CTR at or above tier average: no action needed, page is performing as expected for its position.
- `INSUFFICIENT_DATA` position_tier is "unknown" (no valid `gsc_avg_position`) or impressions too low to trust the CTR estimate; excluded from scoring rather than falsely flagged.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
import os
tier_avg_ctr = df.groupby("position_tier")["ctr"].transform("mean")
df["ctr_gap"] = tier_avg_ctr - df["ctr"]  # positive = underperforming its tier

median_impressions = df["gsc_impressions"].median()
df["high_volume"] = df["gsc_impressions"] >= median_impressions

def assign_reason_code(row):
    if row["position_tier"] == "unknown" or row["gsc_impressions"] < 10:
        return "INSUFFICIENT_DATA"
    elif row["ctr_gap"] <= 0:
        return "AT_OR_ABOVE_EXPECTED"
    elif row["high_volume"]:
        return "LOW_CTR_HIGH_VOLUME"
    else:
        return "LOW_CTR_LOW_VOLUME"

df["reason_code"] = df.apply(assign_reason_code, axis=1)

df["log_impressions"] = np.log1p(df["gsc_impressions"])
df["opportunity_score"] = np.where(
    df["reason_code"].isin(["LOW_CTR_HIGH_VOLUME", "LOW_CTR_LOW_VOLUME"]),
    df["ctr_gap"] * df["log_impressions"],
    0.0
)

def assign_action(reason):
    if reason == "LOW_CTR_HIGH_VOLUME":
        return "review_now"
    elif reason == "LOW_CTR_LOW_VOLUME":
        return "review_later"
    elif reason == "AT_OR_ABOVE_EXPECTED":
        return "no_action"
    else:
        return "insufficient_data"

df["action"] = df["reason_code"].apply(assign_action)

queue = df.sort_values("opportunity_score", ascending=False)[
    ["client_hash_id", "content_hash_id", "report_date", "position_tier",
     "content_type", "ctr", "ctr_gap", "gsc_impressions", "reason_code",
     "action", "opportunity_score"]
].reset_index(drop=True)

print(queue.head(10))
print("\nReason code distribution:")
print(queue["reason_code"].value_counts())

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("\nWritten to work/outputs/baseline_action_score.csv")


            client_hash_id           content_hash_id report_date  \
0  client_23a62021009f63c4  content_44f34c0a90047651  2026-03-28   
1  client_62f4a7e64f5e0096  content_34a70fea29d15f24  2026-03-04   
2  client_73cda7b4e4f265ea  content_fec55986a1868d62  2026-03-30   
3  client_23a62021009f63c4  content_44f34c0a90047651  2026-03-27   
4  client_73cda7b4e4f265ea  content_fec55986a1868d62  2026-03-31   
5  client_73cda7b4e4f265ea  content_9c057b66c30a3abb  2026-03-02   
6  client_73cda7b4e4f265ea  content_9c057b66c30a3abb  2026-03-01   
7  client_23a62021009f63c4  content_44f34c0a90047651  2026-03-25   
8  client_23a62021009f63c4  content_44f34c0a90047651  2026-03-29   
9  client_23a62021009f63c4  content_44f34c0a90047651  2026-03-24   

  position_tier     content_type       ctr   ctr_gap  gsc_impressions  \
0         top_3  keyword article  0.000025  0.004731            40084   
1         top_3  keyword article  0.000051  0.004704            39003   
2         top_3  keyword article

In [5]:
print(df[df["reason_code"] == "INSUFFICIENT_DATA"]["gsc_impressions"].describe())
print(df[df["reason_code"] == "INSUFFICIENT_DATA"]["position_tier"].value_counts())

count    1.463532e+06
mean     3.658095e+00
std      2.497452e+00
min      1.000000e+00
25%      1.000000e+00
50%      3.000000e+00
75%      5.000000e+00
max      9.000000e+00
Name: gsc_impressions, dtype: float64
position_tier
page_1      500442
top_3       339391
page_3_5    246634
deep        216042
striking    161023
Name: count, dtype: int64


INSUFFICIENT_DATA also catches impressions < 10, not just unknown tier, a low-impression page's CTR estimate is too noisy to trust even if position is known.

The score is 0 for anything not flagged, so the ranked queue naturally puts all real opportunities at the top and everything else below.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
print(queue.head(20))

             client_hash_id           content_hash_id report_date  \
0   client_23a62021009f63c4  content_44f34c0a90047651  2026-03-28   
1   client_62f4a7e64f5e0096  content_34a70fea29d15f24  2026-03-04   
2   client_73cda7b4e4f265ea  content_fec55986a1868d62  2026-03-30   
3   client_23a62021009f63c4  content_44f34c0a90047651  2026-03-27   
4   client_73cda7b4e4f265ea  content_fec55986a1868d62  2026-03-31   
5   client_73cda7b4e4f265ea  content_9c057b66c30a3abb  2026-03-02   
6   client_73cda7b4e4f265ea  content_9c057b66c30a3abb  2026-03-01   
7   client_23a62021009f63c4  content_44f34c0a90047651  2026-03-25   
8   client_23a62021009f63c4  content_44f34c0a90047651  2026-03-29   
9   client_23a62021009f63c4  content_44f34c0a90047651  2026-03-24   
10  client_23a62021009f63c4  content_44f34c0a90047651  2026-03-26   
11  client_73cda7b4e4f265ea  content_9c057b66c30a3abb  2026-03-03   
12  client_e547b89c05043229  content_757b1fa67827358d  2026-03-13   
13  client_73cda7b4e4f265ea  conte

# **Distinct pages in this list:**

7 (content_44f34c0a90047651 x 6,
content_fec55986a1868d62 x 3, content_9c057b66c30a3abb x 3,
content_8e1334d6356668e3 x 4, content_8d7d99f109e19aa2 x 2,
content_34a70fea29d15f24 x 1, content_757b1fa67827358d x 1).

Every row shares the same reason_code (LOW_CTR_HIGH_VOLUME) and action (review_now) this baseline's top-20 is really a "top 7 pages, reviewed on repeat" list.

The clearest, most honest finding here: 6 out of 7 distinct pages have exact-zero CTR despite tens of thousands of impressions.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

# **Weak picks**
No. 17 and 20: CTR 0.000139 and 0.000179, the only non-zero CTRs in the top 20. These are the least severe cases in the list, yet they're still ranked "review_now" alongside pages with literal zero CTR. This suggests the score formula (gap x log(impressions)) may be too forgiving of a small gap when volume is high.

Rows #8-11, #15-16, #18-19 (repeated pages) six distinct pages account for all 20 rows, several appearing 3-6 times each. This isn't a wrong pick in the sense of a bad score, but it's a wrong "design": a reviewer working this list would see the same page recommendation repeated across multiple days, wasting review time. The baseline should aggregate to one row per page before ranking, not rank at page-day grain.

Pages with exact zero CTR at 15K-40K impressions (5 of 7 distinct pages) zero clicks across tens of thousands of impressions is unusual enough that it reads more like a possible tracking/attribution gap for that specific page than a genuine "bad title" story. Flagging these as content problems without first checking for a data issue risks sending a reviewer to fix something that isn't actually broken in the way assumed.

# **Leakage check**

No product flags used.The rule only uses `gsc_avg_position`,
  `gsc_impressions`, `gsc_clicks` ( `ctr`), and `content_type`
  
All raw warehouse fields, none of own computed flags or scores (which the release doesn't ship anyway, per the lane guide).


No future window used. All data comes from `month=2026-03` only; the score for a given page-day uses only that day's own `ctr`/`impressions`/`position`,
never a later date's outcome. The sealed final month (June 2026) was never touched.

No label-derived inputs.`ctr_gap` here is computed from the raw CTR and the tier average CTR within the same month, it's a rule input, not something derived from a label I'm trying to predict.

This rule doesn't use any model or Label at all; it's a hand-encoded
rule, so there's no label to leak from in the first place.

## Self-check

- Every section above is filled — markdown thinking AND the code that backs it
- The notebook runs top to bottom with no errors (Runtime → Run all)
- No client names, URLs, or private queries anywhere
- My claims use careful words: observed, measured, directional, decision-support
- Committed to my repo under `work/notebooks/`